# Predicción de subscripción a un producto bancario

## 1. EDA

Para hacer un EDA, debemos usar pickle para deserializar los datos.

### 1.1. Instalación de pickle y pandas

In [1]:
%pip install pandas 

Note: you may need to restart the kernel to use updated packages.


### 1.2. Deserialización de los datos

Con la librería pickle, extraemos los datos de forma sencilla. 

In [2]:
import pickle as pkl
import pandas as pd
import numpy as np
import os

#Hay que aññadir la semilla

file = "./dataset/bank_10.pkl"

if os.path.exists(file):
    with open(file, 'rb') as fd:
        df = pkl.load(fd)
        print(df)


       age          job  marital  education default  balance housing loan  \
0       59       admin.  married  secondary      no     2343     yes   no   
1       56         None  married  secondary      no       45      no   no   
2       41   technician  married  secondary      no     1270     yes   no   
3       55         None  married  secondary      no     2476     yes   no   
4       54       admin.     None   tertiary      no      184      no   no   
...    ...          ...      ...        ...     ...      ...     ...  ...   
11157   33  blue-collar   single    primary      no        1     yes   no   
11158   39     services  married  secondary      no      733      no   no   
11159   32   technician   single  secondary      no       29      no   no   
11160   43   technician  married  secondary      no        0      no  yes   
11161   34   technician  married  secondary      no        0      no   no   

        contact  day month  duration  campaign  pdays  previous poutcome  \

C:\Users\Sergio\AppData\Local\Temp\ipykernel_13776\4200804693.py:12: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  df = pkl.load(fd)


### 1.3. Análisis de variables e instancias

A continuación, analizaremos tanto el número de variables y de instancias, así como sus principales características. 

En cuanto al número de variables y de instancias se puede observar fácilmente de la siguiente forma. 

In [3]:
n_vars = len(df.columns) # Número de variables
n_instances = len(df) # Número de instancias

print(f"Número de variables: {n_vars}\nNúmero de instancias: {n_instances}")


Número de variables: 17
Número de instancias: 11000


Antes de continuar, para evitar futuros problemas, reemplazaremos posibles textos que puedan ser malinterpretados como NaN. 

In [4]:
# Reemplazar valores que representan faltantes
df = df.replace(['None', 'null', 'NaN', 'N/A', ''], np.nan)

Debemos identificar si es un problema de clasificación o regresión, y en caso de ser clasificación, analizar el desbalanceo.

In [ ]:
target_col = 'deposit'

print("Tipo de problema:")
if df[target_col].dtype == 'object' or df[target_col].nunique() < 10:
    print("Clasificación binaria")
else:
    print("Regresión")

print("\nDistribución de la variable objetivo:")
print(df[target_col].value_counts())

print("\nPorcentaje:")
print(df[target_col].value_counts(normalize=True) * 100)

0        yes
1        yes
2        yes
3        yes
4        yes
        ... 
11157     no
11158     no
11159     no
11160     no
11161     no
Name: deposit, Length: 11000, dtype: object
Tipo de problema:
Clasificación binaria

Distribución de la variable objetivo:
deposit
no     5780
yes    5220
Name: count, dtype: int64

Porcentaje:
deposit
no     52.545455
yes    47.454545
Name: proportion, dtype: float64


Sabiendo que el número de variables es 17, nos gustaría saber cuántas de ellas son numéricas, cuántas son categóricas y cuántas son ordinales. 

In [6]:
num_vars = df.select_dtypes(include=['number']).columns.tolist() # Variables numéricas

card_ord_vars = df.select_dtypes(include=['object', 'category']) # Variables categóricas y ordinales

# Ahora debemos distinguir cuáles son categóricas y cuáles son ordinales
for var in card_ord_vars.columns:
    print(f"Variable {var} tiene los valores {df[var].unique()}")

Variable job tiene los valores ['admin.' None 'technician' 'management' 'retired' 'services'
 'blue-collar' 'unemployed' 'entrepreneur' 'housemaid' 'unknown'
 'self-employed' 'student']
Variable marital tiene los valores ['married' None 'single' 'divorced']
Variable education tiene los valores ['secondary' 'tertiary' 'primary' 'unknown']
Variable default tiene los valores ['no' 'yes']
Variable housing tiene los valores ['yes' 'no']
Variable loan tiene los valores ['no' 'yes']
Variable contact tiene los valores ['unknown' 'cellular' 'telephone']
Variable month tiene los valores ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'jan' 'feb' 'mar' 'apr' 'sep']
Variable poutcome tiene los valores ['unknown' 'other' 'failure' 'success']
Variable deposit tiene los valores ['yes' 'no']


Viendo los valores, podemos entonces ver que hay algunas de ellas que mantienen un orden, y por tanto, son ordinales. 

Estas variables son:

Education

Month

Por tanto, el resto serán categóricas

In [7]:
ord_vars = [] # Añadimos las variables ordinales a la lista
for var in card_ord_vars.columns:
    if var == 'education':  # primary < secondary < tertiary
        ord_vars.append(var)
    elif var == 'month':  # orden temporal
        ord_vars.append(var)

# El resto son categóricas
cat_vars = [var for var in card_ord_vars.columns if var not in ord_vars]

print(f"Variables numéricas: {", ".join(num_vars) if num_vars else "No se encontraron variables numéricas"}")
print(f"Variables categóricas: {", ".join(cat_vars) if cat_vars else "No se encontraron variables categóricas"}")
print(f"Variables ordinales: {", ".join(ord_vars) if ord_vars else "No se encontraron variables ordinales"}")

Variables numéricas: age, balance, day, duration, campaign, pdays, previous
Variables categóricas: job, marital, default, housing, loan, contact, poutcome, deposit
Variables ordinales: education, month


Ahora que ya hemos distinguido entre variables categóricas y ordinales, debemos analizar su cardinalidad

In [8]:
cat_hcard_var = [] # Variables categóricas con alta cardinalidad
ord_hcard_var = [] # Variables ordinales con alta cardinalidad

# Vamos a definir una variable de alta cardinalidad cuando hay más de 10 valores únicos
for var in cat_vars:
    if df[var].nunique() > 10:
        cat_hcard_var.append(var)

for var in ord_vars:
    if df[var].nunique() > 10:
        cat_hcard_var.append(var)

print(f"Variables categóricas con alta cardinalidad: {", ".join(cat_hcard_var) if cat_hcard_var else "No se encontraron elementos"}")
print(f"Variables ordinales con alta cardinalidad: {", ".join(ord_hcard_var) if ord_hcard_var else "No se encontraron elementos"}")

Variables categóricas con alta cardinalidad: job, month
Variables ordinales con alta cardinalidad: No se encontraron elementos


Por tanto, a modo de resumen, tenemos

In [9]:
print(f"Variables númericas: {", ".join(num_vars) if num_vars else "No se encontraron elementos"}")
print("_________________________________________________________________________________________")
print(f"Variables categóricas: {", ".join(cat_vars) if cat_vars else "No se encontraron elementos"}")
print(f"\tDe las cuales con alta cardinalidad: {", ".join(cat_hcard_var) if cat_hcard_var else "No se encontraron elementos"}")
print("_________________________________________________________________________________________")
print(f"Variables númericas: {", ".join(ord_vars) if ord_vars else "No se encontraron elementos"}")
print(f"\tDe las cuales con alta cardinalidad: {", ".join(ord_hcard_var) if ord_hcard_var else "No se encontraron elementos"}")

Variables númericas: age, balance, day, duration, campaign, pdays, previous
_________________________________________________________________________________________
Variables categóricas: job, marital, default, housing, loan, contact, poutcome, deposit
	De las cuales con alta cardinalidad: job, month
_________________________________________________________________________________________
Variables númericas: education, month
	De las cuales con alta cardinalidad: No se encontraron elementos


Ahora que tenemos una idea más clara de cómo son las variables, debemos analizar cuales de ellas tienen valores faltantes y, en ese caso, cuánta cantidad de ellos

In [ ]:
# Creamos una lista de tuplas donde cada elemento es una tupla de dos elementos
# Primer elemento: variable
# Segundo elemento: número de valores faltantes
missing_values_count = []

for col in df.columns:
    n_null = df[col].isnull().sum() # Contamos el número de valores faltantes
    missing_values_count.append((col, int(n_null)))

    if n_null > 0:
        print(f"La variable {col} tiene {n_null} valores faltantes")
    else:
        print(f"La variable {col} no tiene valores faltantes")

La variable age no tiene valores faltantes
La variable job tiene 332 valores faltantes
La variable marital tiene 183 valores faltantes
La variable education no tiene valores faltantes
La variable default no tiene valores faltantes
La variable balance no tiene valores faltantes
La variable housing no tiene valores faltantes
La variable loan no tiene valores faltantes
La variable contact no tiene valores faltantes
La variable day no tiene valores faltantes
La variable month no tiene valores faltantes
La variable duration no tiene valores faltantes
La variable campaign no tiene valores faltantes
La variable pdays no tiene valores faltantes
La variable previous no tiene valores faltantes
La variable poutcome no tiene valores faltantes
La variable deposit no tiene valores faltantes
[('age', 0), ('job', 332), ('marital', 183), ('education', 0), ('default', 0), ('balance', 0), ('housing', 0), ('loan', 0), ('contact', 0), ('day', 0), ('month', 0), ('duration', 0), ('campaign', 0), ('pdays', 0)

### 1.X. Visualización de los datos

Para poder visualizar mejor los datos y hacer un análisis de ellos, usaremos streamlit.

In [11]:
!pip install streamlit

De esta forma, podremos ver de forma más visual algunas propiedades de los datos.

In [12]:
%%writefile visualizer.py
import pandas as pd
import streamlit as st

import pickle as pkl

file = "./dataset/bank_10.pkl"

with open(file, 'rb') as fd:
    df = pkl.load(fd)
    print(df)

# Extract the number of variables and instances
df_shape = df.shape

# Create a chart
st.subheader("Número de instancias y variables")
st.table(df_shape, columns=["Instancias", "Variables"])


Overwriting visualizer.py


### 1.4 Correr la aplicación

Para visualizar los datos, ejecutamos el siguiente comando:

In [13]:
!streamlit run visualizer.py

^C


### 1.4. Detección de columnas constantes o IDs

Debemos verificar si hay columnas que tengan un solo valor único (constantes) o que sean posibles identificadores.

In [14]:
# Columnas constantes (un solo valor único)
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]

print("Columnas constantes (1 valor único):")
if constant_cols:
    for col in constant_cols:
        print(f"  - {col}: {df[col].unique()[0]}")
else:
    print("  No hay columnas constantes.")

# Posibles IDs (valores únicos = número de filas)
potential_ids = [col for col in df.columns if df[col].nunique() == len(df)]

print("\nPosibles columnas de ID (todos los valores únicos):")
if potential_ids:
    print(potential_ids)
else:
    print("  No hay columnas que parezcan IDs.")

Columnas constantes (1 valor único):
  No hay columnas constantes.

Posibles columnas de ID (todos los valores únicos):
  No hay columnas que parezcan IDs.


### 1.5. Análisis especial de la variable `pdays`

Según la descripción del dataset:
- `pdays`: Número de días desde que el cliente fue contactado en la campaña anterior
- **Valor -1 significa: sin contacto o desconocido**

Esta variable requiere un análisis y preprocesamiento especial.

In [15]:
print("="*60)
print("ANÁLISIS DE LA VARIABLE 'pdays'")
print("="*60)

if 'pdays' in df.columns:
    print("\nEstadísticas descriptivas de 'pdays':")
    print(df['pdays'].describe())
    
    # Contar valores -1
    n_minus_one = (df['pdays'] == -1).sum()
    pct_minus_one = (n_minus_one / len(df) * 100)
    
    print(f"\nValores = -1 (sin contacto previo): {n_minus_one} ({pct_minus_one:.2f}%)")
    print(f"Valores > 0 (con contacto previo): {(df['pdays'] > 0).sum()}")
    
    # Distribución de valores
    print(f"\nDistribución de valores:")
    print(f"  Mínimo: {df['pdays'].min()}")
    print(f"  Máximo: {df['pdays'].max()}")
    
    if (df['pdays'] > 0).sum() > 0:
        print(f"  Media (sin -1): {df[df['pdays'] > 0]['pdays'].mean():.2f}")
        print(f"  Mediana (sin -1): {df[df['pdays'] > 0]['pdays'].median():.2f}")

ANÁLISIS DE LA VARIABLE 'pdays'

Estadísticas descriptivas de 'pdays':
count    11000.000000
mean        51.308636
std        108.782842
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         20.250000
max        854.000000
Name: pdays, dtype: float64

Valores = -1 (sin contacto previo): 8203 (74.57%)
Valores > 0 (con contacto previo): 2797

Distribución de valores:
  Mínimo: -1
  Máximo: 854
  Media (sin -1): 204.72
  Mediana (sin -1): 182.00


#### Estrategia de preprocesamiento para `pdays`

Dado que -1 representa "sin contacto previo" y aparece en un porcentaje significativo de los datos, la mejor estrategia es:

1. **Crear una variable binaria** `fue_contactado` que indique si el cliente fue contactado previamente (1) o no (0)
2. **Reemplazar los valores -1** en `pdays` por 0 para mantener la variable numérica válida

Esta estrategia permite que los modelos capturen tanto:
- **El hecho de haber sido contactado** (información categórica)
- **Cuánto tiempo hace del contacto** (información numérica)


In [16]:
# Implementar preprocesamiento de pdays
print("Aplicando preprocesamiento a 'pdays'...\n")

# Crear variable binaria
df['fue_contactado'] = (df['pdays'] > 0).astype(int)

# Reemplazar -1 con 0
df['pdays'] = df['pdays'].replace(-1, 0)

print("✓ Variable 'fue_contactado' creada:")
print(df['fue_contactado'].value_counts())

print("\n✓ Variable 'pdays' preprocesada (valores -1 → 0):")
print(f"  Valores únicos: {df['pdays'].nunique()}")
print(f"  Rango: [{df['pdays'].min()}, {df['pdays'].max()}]")

print("\nPreprocesamiento de 'pdays' completado.")

Aplicando preprocesamiento a 'pdays'...

✓ Variable 'fue_contactado' creada:
fue_contactado
0    8203
1    2797
Name: count, dtype: int64

✓ Variable 'pdays' preprocesada (valores -1 → 0):
  Valores únicos: 472
  Rango: [0, 854]

Preprocesamiento de 'pdays' completado.


### 1.6. Resumen del EDA

A continuación se presenta una tabla resumen con todos los hallazgos del análisis exploratorio:

In [17]:
# Calcular métricas para el resumen
n_categoricas = len(cat_vars)
n_numericas = len(num_vars)
n_ordinales = len(ord_vars)
n_alta_card = len(cat_hcard_var) + len(ord_hcard_var)
n_faltantes = len([x for x in missing_values_count if x[1] > 0])

print("="*60)
print("RESUMEN DEL EDA")
print("="*60)
print(f"\nDimensiones del dataset:")
print(f"  - Instancias: {n_instances}")
print(f"  - Variables (original): {n_vars}")
print(f"  - Variables (tras preprocesar pdays): {len(df.columns)}")

print(f"\nTipos de variables:")
print(f"  - Variables numéricas: {n_numericas}")
print(f"  - Variables categóricas: {n_categoricas}")
print(f"  - Variables ordinales: {n_ordinales}")

print(f"\nCaracterísticas especiales:")
print(f"  - Variables con alta cardinalidad (>10): {n_alta_card}")
if cat_hcard_var or ord_hcard_var:
    print(f"    Específicamente: {', '.join(cat_hcard_var + ord_hcard_var)}")
print(f"  - Variables con valores faltantes: {n_faltantes}")
if n_faltantes > 0:
    vars_faltantes = [x[0] for x in missing_values_count if x[1] > 0]
    print(f"    Específicamente: {', '.join(vars_faltantes)}")
print(f"  - Columnas constantes: {len(constant_cols)}")
print(f"  - Columnas de ID: {len(potential_ids)}")

print(f"\nProblema de ML:")
print(f"  - Tipo: Clasificación binaria")
print(f"  - Variable objetivo: deposit")
print(f"  - Distribución de clases: {df['deposit'].value_counts().to_dict()}")

# Calcular si está desbalanceado
class_pct = df['deposit'].value_counts(normalize=True) * 100
minority_pct = class_pct.min()
if minority_pct < 40:
    print(f"  - Desbalanceo: SÍ (clase minoritaria: {minority_pct:.2f}%)")
else:
    print(f"  - Desbalanceo: NO (clase minoritaria: {minority_pct:.2f}%)")

RESUMEN DEL EDA

Dimensiones del dataset:
  - Instancias: 11000
  - Variables (original): 17
  - Variables (tras preprocesar pdays): 18

Tipos de variables:
  - Variables numéricas: 7
  - Variables categóricas: 8
  - Variables ordinales: 2

Características especiales:
  - Variables con alta cardinalidad (>10): 2
    Específicamente: job, month
  - Variables con valores faltantes: 2
    Específicamente: job, marital
  - Columnas constantes: 0
  - Columnas de ID: 0

Problema de ML:
  - Tipo: Clasificación binaria
  - Variable objetivo: deposit
  - Distribución de clases: {'no': 5780, 'yes': 5220}
  - Desbalanceo: NO (clase minoritaria: 47.45%)
